# 02 — Classify a SEC filing

Runs the classifier on a sample PDF and shows the per-segment category map.
The classifier delegates to the analyzer via
`contentCategories[<cat>].analyzerId`, so a single `begin_analyze_binary`
call exercises both stages.

Mirrors `sec_service.classify_and_extract()` — the `cu_classify_and_extract`
step shown live on the SEC page.


In [ ]:
from _lib import sec_service
samples = sec_service.list_samples()
for s in samples:
    print(f"  {s['file_name']:55s}  {s['size_kb']:8.1f} KB  cached={s['has_cached_result']}")

sample = samples[0]['file_name']
print('\nUsing:', sample)

## Run the classifier + analyzer (one CU call)

In [ ]:
from pathlib import Path
sec_service.ensure_analyzers()
pdf_bytes = (sec_service.ATTACH_DIR / sample).read_bytes()
raw, retries = sec_service.classify_and_extract(pdf_bytes)
print(f'retries: {retries}')
print('segment categories:', sec_service.segment_category_counts(raw))
print('missing statements:', sec_service.missing_categories(raw))

## Inspect the per-segment summary

In [ ]:
for seg in raw.get('contents', []):
    cat = seg.get('category', 'Unknown')
    if cat == 'Other':
        continue
    ft = (seg.get('fields') or {}).get('financialTables', {}) or {}
    tables = ft.get('valueArray', []) or []
    pg_start = seg.get('startPageNumber', '?')
    pg_end = seg.get('endPageNumber', '?')
    print(f"  {cat:22s} pp.{pg_start}-{pg_end}  tables={len(tables)}")